In [0]:
%run ../07_Common/00_setup

In [0]:
df_silver_products = spark.table("workspace.silver.products")

df_dim_products = (
    df_silver_products
    .select(
        F.col("id").alias("product_id"),
        F.col("title").alias("product_name"),
        F.col("category"),
        F.col("brand"),
        F.col("price").cast("decimal(10,2)").alias("price"),
        F.col("discount_percentage").cast("decimal(5,2)").alias("discount_percentage"),
        F.col("rating").cast("decimal(3,2)").alias("rating"),
        F.col("stock").cast("int").alias("stock"),
    )
)

print(f" Registros en dim_products: {df_dim_products.count()}")

In [0]:
# ============================================================
# CELDA 3 — Escribir en Gold (overwrite completo, idempotente)
# ============================================================
try:
    (
        df_dim_products.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("workspace.gold.dim_products")
    )
    estado_final = "success"
    print("Escritura exitosa en gold.dim_products")
except Exception as e:
    estado_final = "failed"
    print(f"ERROR: {e}")

spark.sql(f"""
    UPDATE workspace.control.gold_aggregation_config
    SET last_run_status = '{estado_final}'
    WHERE entity_name = 'dim_products'
""")
dbutils.notebook.exit(f"{estado_final.upper()} | dim_products")